# Retrieval-Augmented PPO on Frostbite (JAX)

Select a GPU runtime. This notebook installs the uv-managed CUDA environment, checks JAX and W&B, runs unit/smoke validation, and launches the matched A/B/C seed matrix. Long-run artifacts can be written directly to Google Drive.


In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

# Set this to your Git URL after pushing MemRL, or upload/copy the project to /content/MemRL.
REPO_URL = ""  # @param {type:"string"}
if REPO_URL:
    !git clone "$REPO_URL" /content/MemRL
%cd /content/MemRL


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv sync --extra cuda12 --extra dev

# Never print the secret. Add WANDB_API_KEY under Colab Secrets first.
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
!uv run python -c "import envpool, jax; print(envpool.__version__, jax.__version__, jax.devices())"
!uv run python -c "import jax; assert any(d.platform == 'gpu' for d in jax.devices())"


In [ ]:
!uv run pytest -q
!uv run memrl-smoke --wandb-mode disabled


## Mandatory performance gate

This runs the nine exclusive 200k-step timing processes, excludes compilation and the first 100k steps, and writes the paired ratio/resource report. Do not launch the learning matrix unless it passes.


In [ ]:
!uv run memrl-benchmark --run-gate --output benchmark-report.json


## Full experiment matrix

The command runs one job at a time to stay within GPU memory: modes `none`, `random`, `learned`, with seeds 1, 2, and 3. Set Drive paths so Colab runtime storage is not the only copy.


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/MemRL/runs"  # @param {type:"string"}
CHECKPOINT_DIR = "/content/drive/MyDrive/MemRL/checkpoints"  # @param {type:"string"}
!uv run memrl-matrix --max-parallel 1 --output-dir "$OUTPUT_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" --wandb-project memrl-frostbite
